# Download OSM layers within a 5 km buffer of the AOI

In [19]:
from pathlib import Path
import geopandas as gpd
import osmnx as ox

ox.settings.cache_folder = "data/cache"

OUT = Path("data/geopackage")
OUT.mkdir(parents=True, exist_ok=True)

## Area of interest: 5 km buffer around the buildings

In [20]:
buildings = gpd.read_file("data/buildings_aoi.gpkg")
aoi_buffer = buildings.to_crs(32721).union_all().convex_hull.buffer(5_000)
aoi_buffer = gpd.GeoSeries([aoi_buffer], crs=32721).to_crs(4326).iloc[0]
gpd.GeoSeries([aoi_buffer], crs=4326).to_file(OUT / "aoi_buffer_5km.gpkg")

In [21]:
def save(gdf, name):
    gdf = gdf.reset_index()
    gdf.to_file(OUT / f"{name}.gpkg")
    print(f"{name}: {len(gdf)} features")

## Green areas

In [15]:
tags = {
    "leisure": ["park", "garden", "nature_reserve", "playground", "recreation_ground"],
    "landuse": ["forest", "grass", "recreation_ground", "village_green", "meadow"],
    "natural": ["wood", "scrub", "grassland"],
    "place": ["square"],
}
green = ox.features_from_polygon(aoi_buffer, tags)
green = green[green.geom_type.isin(["Polygon", "MultiPolygon"])]
save(green, "green_areas")

green_areas: 1271 features


## Public transportation nodes and hubs

In [22]:
tags = {"highway": "bus_stop", "amenity": "bus_station"}
save(ox.features_from_polygon(aoi_buffer, tags), "public_transport")

public_transport: 230 features


## Amenities

In [23]:
tags = {"amenity": ["restaurant", "cafe", "bar"]}
save(ox.features_from_polygon(aoi_buffer, tags), "amenities")

amenities: 185 features


## Coastlines and beaches

In [24]:
tags = {"natural": ["beach", "coastline"], "leisure": "beach_resort"}
save(ox.features_from_polygon(aoi_buffer, tags), "coast_beaches")

coast_beaches: 66 features


## Road / street network (all modes)

In [25]:
G = ox.graph_from_polygon(aoi_buffer, network_type="all")
ox.save_graph_geopackage(G, OUT / "street_network.gpkg")
print(f"{len(G.nodes)} nodes, {len(G.edges)} edges")

9070 nodes, 24062 edges
